In [1]:
# Install system dependency for PDF conversion
!apt-get install -y poppler-utils

# Install Python dependencies
!pip install pdf2image python-docx easyocr google-genai openai jiwer opencv-python-headless matplotlib pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [2]:
import os
import cv2
import numpy as np
import torch
import easyocr
import jiwer
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import google.generativeai as genai
from google.colab import drive
from google.colab import userdata

# 1. Mount Google Drive
drive.mount('/content/drive')


# 3. Define Directories (pointing to your new clean structure)
BASE_PATH = '/content/drive/MyDrive/RenAIssance_GSoC/'
IMG_DIR = os.path.join(BASE_PATH, 'outputs/extracted_images/')
GT_DIR = os.path.join(BASE_PATH, 'ground_truth/clean_text/')
OUT_DIR = os.path.join(BASE_PATH, 'outputs/ocr_results/')
os.makedirs(OUT_DIR, exist_ok=True)

print("Setup complete. Directories mapped.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete. Directories mapped.


In [3]:
print("Initializing models (this may take a minute)...")

# 1. Load TrOCR (for reading the text)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
trocr_model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed').to(device)
print(f"TrOCR loaded successfully on: {device}")

# 2. Load EasyOCR (Used ONLY for finding line coordinates, not predicting text)
reader = easyocr.Reader(['es'], gpu=True)
print("EasyOCR loaded successfully.")

Initializing models (this may take a minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TrOCR loaded successfully on: cuda
EasyOCR loaded successfully.


In [5]:
from openai import OpenAI
import time

def crop_main_text(pil_image):
    """Isolates the main text block, removing marginalia using OpenCV."""
    img = np.array(pil_image.convert('L'))
    _, thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Dilate to form a single solid block
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 40))
    dilated = cv2.dilate(thresh, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours: return pil_image

    # Find the largest contour
    main_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(main_contour)

    # Add padding and crop
    pad = 20
    x, y = max(0, x-pad), max(0, y-pad)
    w, h = min(img.shape[1]-x, w+(pad*2)), min(img.shape[0]-y, h+(pad*2))
    return pil_image.crop((x, y, x + w, y + h))

def run_trocr_pipeline(pil_image):
    """Hybrid approach: EasyOCR for segmentation, TrOCR for recognition."""
    img_np = np.array(pil_image)

    # Get bounding boxes for text lines
    results = reader.readtext(img_np, width_ths=0.7)
    sorted_boxes = sorted(results, key=lambda x: x[0][0][1])

    extracted_lines = []
    for bbox, _, _ in sorted_boxes:
        # Extract coordinates for each line
        x_coords, y_coords = [int(pt[0]) for pt in bbox], [int(pt[1]) for pt in bbox]
        x_min, x_max = max(0, min(x_coords)), max(x_coords)
        y_min, y_max = max(0, min(y_coords)), max(y_coords)

        # Crop the specific line
        pad = 2
        line_img = pil_image.crop((x_min, max(0, y_min-pad), x_max, y_max+pad))

        # Run TrOCR on the single line
        pixel_values = processor(line_img.convert("RGB"), return_tensors="pt").pixel_values.to(device)
        generated_ids = trocr_model.generate(pixel_values)
        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        extracted_lines.append(text)

    return "\n".join(extracted_lines)

client = OpenAI(
  base_url="https://go.fastrouter.ai/api/v1", # Or fastrouter.ai if that is the one
  api_key=userdata.get('ROUTER_API_KEY'), # Add your new key to Colab Secrets
)

def correct_with_llm(raw_text, retries=3):
    """Uses Claude Sonnet 4.6 via FastRouter for high-accuracy historical correction."""

    # Transcription rules for RenAIssance Project
    system_rules = (
        "You are an expert in 17th-century Spanish philology. "
        "Correct OCR artifacts while following these strict transcription rules: "
        "1. 'u', 'v', and 'b' are used interchangeably; preserve original letters.\n"
        "2. 'i' is often used instead of 'j' (e.g., 'Iacinto', 'Iuan').\n"
        "3. Long-s 'ſ' is often misread as 'f', 'l', or 't'; fix this contextually.\n"
        "4. 'ç' must always be modern 'z' (e.g., 'cobrança' -> 'cobranza').\n"
        "5. Expand abbreviated 'q' with an accent into the full word 'Que'.\n"
        "6. Do NOT modernize spelling; keep historical forms like 'auia'.\n"
        "7. Output ONLY the corrected text."
    )

    for attempt in range(retries):
        try:
            # Using the Sonnet 4.6 identifier
            response = client.chat.completions.create(
                model="anthropic/claude-sonnet-4.6",
                messages=[
                    {"role": "system", "content": system_rules},
                    {"role": "user", "content": f"RAW OCR: {raw_text}"}
                ],
                temperature=0.1 # Low temperature for strict rule-following
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"      [FastRouter Error: {e}. Retrying {attempt + 1}/{retries}...]")
            time.sleep(2)

    return raw_text

In [6]:
print("\nStarting Evaluation Loop...\n")
total_cer = 0
total_wer = 0
processed_count = 0

# Loop through your clean ground truth text files
for txt_filename in sorted(os.listdir(GT_DIR)):
    if not txt_filename.endswith('.txt'):
        continue

    # Construct the matching image filename
    base_name = txt_filename.replace('.txt', '')
    img_filename = base_name + '.png'

    img_path = os.path.join(IMG_DIR, img_filename)
    txt_path = os.path.join(GT_DIR, txt_filename)

    if not os.path.exists(img_path):
        print(f"Warning: Image {img_filename} not found. Skipping.")
        continue

    print(f"Processing: {base_name}")

    # 1. Read the ground truth text
    with open(txt_path, 'r', encoding='utf-8') as f:
        ground_truth = f.read().strip()

    # 2. Read the image
    image = Image.open(img_path)

    # 3. Execute the pipeline
    cropped_img = crop_main_text(image)
    raw_ocr = run_trocr_pipeline(cropped_img)
    final_text = correct_with_llm(raw_ocr)

    output_filename = f"{base_name}_predicted.txt"
    output_path = os.path.join(OUT_DIR, output_filename)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(final_text)

    # 4. Calculate error rates
    gt_clean = " ".join(ground_truth.lower().split())
    pred_clean = " ".join(final_text.lower().split())
    cer = jiwer.cer(gt_clean, pred_clean)
    wer = jiwer.wer(gt_clean, pred_clean)

    total_cer += cer
    total_wer += wer
    processed_count += 1

    print(f"  -> CER: {cer:.4f} | WER: {wer:.4f}\n")

# Print the final averaged results
if processed_count > 0:
    print("========================================")
    print(f"FINAL METRICS (Averaged over {processed_count} pages)")
    print(f"Average CER: {total_cer / processed_count:.4f}")
    print(f"Average WER: {total_wer / processed_count:.4f}")
    print("========================================")
else:
    print("No valid image/text pairs found to process. Check your directories.")


Starting Evaluation Loop...

Processing: Buendia - Instruccion_page_2
  -> CER: 0.2358 | WER: 0.3274

Processing: Buendia - Instruccion_page_3
  -> CER: 0.5238 | WER: 0.5617

Processing: Buendia - Instruccion_page_4
  -> CER: 0.4459 | WER: 0.6011

Processing: Covarrubias - Tesoro lengua_page_7
  -> CER: 0.3867 | WER: 0.5213

Processing: Covarrubias - Tesoro lengua_page_8
  -> CER: 0.4654 | WER: 0.6211

Processing: Covarrubias - Tesoro lengua_page_9
  -> CER: 0.5257 | WER: 0.7433

Processing: Guardiola - Tratado nobleza_page_12
  -> CER: 0.1523 | WER: 0.3500

Processing: Guardiola - Tratado nobleza_page_13
  -> CER: 0.1219 | WER: 0.3256

Processing: Guardiola - Tratado nobleza_page_14
  -> CER: 0.0403 | WER: 0.1429

Processing: PORCONES.228.38 – 1646_page_1


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (94080000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  -> CER: 0.1696 | WER: 0.3274

Processing: PORCONES.228.38 – 1646_page_2
  -> CER: 0.9864 | WER: 0.9967

Processing: PORCONES.228.38 – 1646_page_3
  -> CER: 0.9850 | WER: 0.9896

Processing: PORCONES.228.38 – 1646_page_4
  -> CER: 0.9747 | WER: 0.9863

Processing: PORCONES.23.5 - 1628_page_1
  -> CER: 0.7521 | WER: 0.9783

Processing: PORCONES.23.5 - 1628_page_2
  -> CER: 0.8948 | WER: 0.9389

Processing: PORCONES.23.5 - 1628_page_3
  -> CER: 0.8604 | WER: 0.9390

Processing: PORCONES.23.5 - 1628_page_4
  -> CER: 0.7586 | WER: 0.9396

Processing: PORCONES.748.6 – 1650_page_1
  -> CER: 0.8160 | WER: 0.9477

Processing: PORCONES.748.6 – 1650_page_2
  -> CER: 0.7153 | WER: 0.8435

Processing: PORCONES.748.6 – 1650_page_3
  -> CER: 0.6800 | WER: 0.8579

Processing: PORCONES.748.6 – 1650_page_4
  -> CER: 0.7605 | WER: 0.9808

FINAL METRICS (Averaged over 21 pages)
Average CER: 0.5834
Average WER: 0.7105
